In [11]:
import pandas as pd
import os
import rasterio
import tqdm
import numpy as np
os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\OGSL')

# Add gebcoDepth

In [6]:
# Step 1: Load the Parquet file
input_file = "Archambault_biodiversity_data_long_2026_01_30.csv"
df = pd.read_csv(input_file, sep=";")
# Step 2: Extract unique station locations
unique_stations = df[["lat_start", "lon_start"]].drop_duplicates()
# Step 3: Save the result to a new file
output_file = "unique_coordinates.csv"
unique_stations.to_csv(output_file, index=False)
print(f"Unique station locations with year saved to {output_file}")

Unique station locations with year saved to unique_coordinates.csv


In [13]:
def extract_raster_values(raster_path, coordinates_df, batch_size=10000):
    """
    Extract raster values for given coordinates from a single raster file.
    Parameters:
        raster_path (str): Path to the merged raster file.
        coordinates_df (pd.DataFrame): DataFrame with 'lat_start' and 'lon_start'.
        batch_size (int): Number of points to process in each batch.
    Returns:
        pd.DataFrame: DataFrame with extracted raster values.
    """
    results = []
    total_batches = len(coordinates_df) // batch_size + (1 if len(coordinates_df) % batch_size != 0 else 0)
    with rasterio.open(raster_path) as src:
        bounds = src.bounds  # Get raster bounds
        with tqdm.tqdm(total=total_batches, desc="Processing Batches", unit="batch") as pbar:
            for i in range(0, len(coordinates_df), batch_size):
                batch = coordinates_df.iloc[i:i + batch_size].copy()
                batch.loc[:, 'gebco'] = np.nan  # Initialize the column for raster values
                # Adjust longitudes at the antimeridian
                batch['lon_start'] = batch['lon_start'].apply(
                    lambda lon: -180 if lon == 180 else lon
                )
                # Filter coordinates within the raster bounds
                batch_in_bounds = batch[
                    (batch['lon_start'] >= bounds.left) &
                    (batch['lon_start'] <= bounds.right) &
                    (batch['lat_start'] >= bounds.bottom) &
                    (batch['lat_start'] <= bounds.top)
                ]
                # Extract raster values for these coordinates
                coords = [(lon, lat) for lon, lat in zip(batch_in_bounds['lon_start'], batch_in_bounds['lat_start'])]
                values = list(src.sample(coords))
                # Assign values to the batch DataFrame
                for idx, value in zip(batch_in_bounds.index, values):
                    if value[0] is not None and not np.isnan(value[0]):
                        batch.loc[idx, 'gebco'] = value[0]
                results.append(batch)
                pbar.update(1)
    return pd.concat(results, ignore_index=True)
# File path to the merged raster
raster_file = r"..\GEBCO\gebco_merged.tif"
# Load the unique_coordinates.csv file
coordinates = pd.read_csv("unique_coordinates.csv")
# Ensure lon_start and lat_start are numeric
coordinates['lon_start'] = pd.to_numeric(coordinates['lon_start'], errors='coerce')
coordinates['lat_start'] = pd.to_numeric(coordinates['lat_start'], errors='coerce')
# Drop rows with invalid (NaN) coordinates
coordinates = coordinates.dropna(subset=['lon_start', 'lat_start'])
# Extract raster values
result_df = extract_raster_values(raster_file, coordinates)
# Save the results to a CSV file
result_df.to_csv("gebco_values.csv", index=False)
print("Results saved as'gebco_values.csv'.")

Processing Batches: 100%|██████████| 1/1 [00:00<00:00, 43.74batch/s]

Results saved as'gebco_values.csv'.


In [16]:
gebco_df = pd.read_csv("gebco_values.csv")
ogsl = pd.read_csv("Archambault_biodiversity_data_long_2026_01_30.csv", sep=";")

ogsl = ogsl.merge(
    gebco_df.rename(columns={"gebco": "gebcoDepth"}),
    on=["lat_start", "lon_start"],
    how="left"
)

ogsl.to_csv("Archambault_biodiversity_data_long_2026_01_30.csv", index=False)
print("Updated Archambault_biodiversity_data_long_2026_01_30.csv with gebcoDepth.")

Updated Archambault_biodiversity_data_long_2026_01_30.csv with gebcoDepth.


# Map to final file

In [ ]:
ogsl = pd.read_csv("Archambault_biodiversity_data_long_2026_01_30.csv", sep=";")

column_map = {
    "datasetKey": "project_id",
    "datasetTitle": "add later",
    "decimalLatitudeStart": "lat_start",
    "decimalLongitudeStart": "lon_start",
    "decimalLatitudeStop": "lat_end",
    "decimalLongitudeStop": "lon_end",
    "depthA": "depth_start",
    "depthB": "depth_end",
    "gebcoDepth": "gebcoDepth",
    "eventID": "eventID",
    "parentEventID": "parentEventID",
    "waterBody": "waterBody",
    "locationID": "locationID",
    "eventDate": "eventDate",
    "startDayOfYear": "startDayOfYear",
    "endDayOfYear": "endDayOfYear",
    "year": "date_year",
    "eventRemarks": "eventRemarks",
    "verbatimScientificName": "originalScientificName",
    "acceptedScientificName": "scientificName",
    "identificationQualifier": "identificationQualifier",
    "lifeStage": "lifeStage",
    "sex": "sex",
    "taxonomicStatus": "taxonomicStatus",
    "taxonRank": "taxonRank",
    "phylum": "phylum",
    "class": "class",
    "order": "order",
    "family": "family",
    "genus": "genus",
    "specificEpithet": "specificEpithet",
    "validAuthority": "scientificNameAuthorship",
    "aphiaID": "aphiaID",
    "individualCount": "individualCount",
    "percentCoverage": "percentCoverage",
    "biomass": "biomass",
    "biomassUnit": "biomassUnit",
    "dynamicProperties": "dynamicProperties",
    "basisOfRecord": "basisOfRecord",
    "gear": "gear",
    "samplingEffort": "samplingEffort",
    "sampleSizeValue": "sampleSizeValue",
    "sampleSizeUnit": "sampleSizeUnit",
}

output_columns = [
    "ID",
    "sourceID",
    "verbatimID",
    "sourceDatabase",
    "datasetKey",
    "datasetTitle",
    "issue",
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "decimalLatitudeStop",
    "decimalLongitudeStop",
    "footprintWKT",
    "coordinateUncertaintyInMeters",
    "hasGeoSpatialIssues",
    "depthA",
    "depthAType",
    "depthB",
    "depthBType",
    "depthAccuracy",
    "gebcoDepth",
    "eventID",
    "parentEventID",
    "waterBody",
    "largeMarineEcosystem",
    "locationID",
    "eventDate",
    "startDayOfYear",
    "endDayOfYear",
    "year",
    "eventRemarks",
    "verbatimScientificName",
    "acceptedScientificName",
    "identificationQualifier",
    "lifeStage",
    "sex",
    "taxonomicStatus",
    "taxonomicIssue",
    "taxonRank",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "specificEpithet",
    "validAuthority",
    "aphiaID",
    "individualCount",
    "samplingEffort",
    "percentCoverage",
    "biomass",
    "biomassUnit",
    "dryMass",
    "dynamicProperties",
    "basisOfRecord",
    "occurrenceRemarks",
    "gear",
    "gearCategory",
    "gearDescription",
    "gearDimensions",
    "meshSize",
    "sieveFraction",
    "sampleSizeValue",
    "sampleSizeUnit",
    "subsampleToSampleRatio",
    "subsampleSpatialLevel",
    "flagged", 
    "flaggedComment"
]

ogsl_final = pd.DataFrame(index=ogsl.index)
missing_sources = []

for col in output_columns:
    if col in column_map:
        src = column_map[col]
        if src in ogsl.columns:
            ogsl_final[col] = ogsl[src]
        else:
            ogsl_final[col] = pd.NA
            missing_sources.append(src)
    else:
        ogsl_final[col] = pd.NA

obis_final = ogsl_final[output_columns]
obis_final.to_parquet("ogsl_final.parquet", index=False)

print(f"Saved ogsl_final.parquet with {len(obis_final):,} rows and {len(obis_final.columns)} columns.")
if missing_sources:
    print("Missing source columns in ogsl:", sorted(set(missing_sources)))

In [ ]:
obis_final = pd.read_parquet("ogsl_final.parquet")
obis_final["sourceDatabase"] = "OGSL"
obis_final.to_parquet("ogsl_final.parquet", index=False)
print("Added sourceDatabase column with value 'OGSL' and saved ogsl_final.parquet.")